# Advanced GLAES Workflow Examples

This notebook introduces several practical GLAES features that build on the basic workflow.
It is intended for users who already understand how to create an `ExclusionCalculator`
and want to explore additional exclusion methods, visualization options, and raster handling.

### Overview:
1. How to inspect region grid properties
2. How to apply exclusion rules from a table using `exclusion_set`
3. How to restart from a precomputed result
4. How to visualize results with basemaps 
5. Why `padExtent` matters near borders
6. How to use `excludeRasterType` with value ranges
7. How to use inclusion mode and inversion
8. How to work with DEM mosaics for elevation and slope exclusions
9. Working with `excludeVectorType()`
10. Visualization methods

In [ ]:
import geokit as gk
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import glaes as gl

In [ ]:
# Use built-in test data so the examples run without extra downloads
region_path = gl._test_data_["aachenShapefile.shp"]
clc_path = gl._test_data_["clc-aachen_clipped.tif"]

# Create a standard exclusion calculator used in several examples
ec = gl.ExclusionCalculator(region_path, srs=3035, pixelRes=100)

# 1. Inspect region grid information

Each `ExclusionCalculator` works on a raster grid defined by the region, spatial reference
system, and pixel resolution.

Understanding this grid is useful because it helps you see:

- the spatial extent being processed
- the raster mask used internally
- the size of the calculation grid

In [ ]:
# Create a fresh ExclusionCalculator so the grid properties are easy to inspect.
ec = gl.ExclusionCalculator(region_path, srs=3035, pixelRes=100)
# The bounding box around the region, expressed in the chosen coordinate system.
ec.region.extent

In [ ]:
# Region mask of the raster grid covering the region's bounding box.
# True marks pixels inside the study region; False marks pixels outside it.
ec.region.mask

In [ ]:
# The shape of the internal grid as (rows, columns).
ec.region.mask.shape

# 2. Applying an Exclusion Set
Multiple exclusion rules can be stored in a CSV file and loaded as a `pandas.DataFrame`. Each row represents one exclusion rule.

In [ ]:
exclusion_set = pd.read_csv(gl._test_data_["sample_exclusion_set.csv"])
exclusion_set

The `name` column identifies the dataset used by a rule, while `type` and `value` define how the exclusion is applied. Optional columns such as `buffer`, `exclusion_mode`, and `invert` can further control each rule.

The datasets referenced in name are passed to `excludeSet` as keyword arguments. For example, **clc** and **osm_road** in the table correspond to the `clc=...` and `osm_roads=...` arguments below.

In [ ]:
# Start with a new ExclusionCalculator for this example.
ec = gl.ExclusionCalculator(region_path, srs=3035, pixelRes=100)

# Apply all exclusion rules listed in the table.
# The additional keyword arguments provide the datasets referenced inside the table.
ec.excludeSet(
    exclusion_set=exclusion_set,
    clc=gl._test_data_["clc-aachen_clipped.tif"],
    osm_roads=gl._test_data_["aachenRoads.shp"],
)

ec.draw()

# 3. Reinitialize an ExclusionCalculator from a pre-computed result
If a previous exclusion result has already been saved as a raster, you can use it as the
starting point for a new `ExclusionCalculator`. **Note:** Reusing a previously saved `.tif` as `initialValue` reproduces the previous availability result exactly only if the new `ExclusionCalculator` uses the same spatial context as the one used to create the file. This includes the same region, SRS, pixel resolution, bounds/extent, and raster alignment. If these settings differ, GLAES will warp the raster to the new calculator context, which provides a similar starting layer but may not be pixel-identical to the original result.

In [ ]:
## Set the initial value to those we computed previously. This is particularly useful if the original calculation took a while
ec = gl.ExclusionCalculator(
    region_path,
    srs=3035,
    pixelRes=100,
    initialValue="data/aachens_best_pv_spots.tif",
)

ec.draw()

# 4. Draw exclusions with a basemap

`drawWithSmopyBasemap()` can be used to display exclusion results on top of a map background. This is useful when the result should be interpreted in its geographic context, for example to compare excluded areas with roads, settlements, or other recognizable landscape features. Keep the zoom level moderate to avoid slow loading.

In [ ]:
ec.drawWithSmopyBasemap?

In [ ]:
ax = ec.drawWithSmopyBasemap(zoom=11, figsize=(40, 20))

# 5. Considering region boundary effects with padExtent 
Features outside the study region can still affect the available area inside it. For example, a road just outside the region boundary may have a buffer that extends into the region.

Without additional padding, features outside the region may not be considered during processing. As a result, exclusions close to the boundary can be incomplete.

`padExtent` extends the area considered during processing beyond the region boundary. The value specifies the additional distance in the units of the calculator's SRS. The final availability, however, is still evaluated only within the original study region.

Below, we compare the processing grid with and without `padExtent`.

In [ ]:
# No padding: the grid covers only the region bounding box.
ec0 = gl.ExclusionCalculator(
    region=region_path,
    srs=3035,
    pixelRes=100,
    padExtent=0,
)

print("Extent (padExtent=0):", ec0.region.extent)
print("Mask shape (padExtent=0):", ec0.region.mask.shape)

In [ ]:
# With padding: the grid is expanded, which helps compute exclusions near the region border correctly.
ec_pad = gl.ExclusionCalculator(
    region=region_path,
    srs=3035,
    pixelRes=100,
    padExtent=2000,  # 2 km padding around the region bounding box
)

print("Extent (padExtent=2000):", ec_pad.region.extent)
print("Mask shape (padExtent=2000):", ec_pad.region.mask.shape)

Note that
- `xMin` and `yMin` become **smaller**
- `xMax` and `yMax` become **larger**

So the box grows outward in every direction, which is why the grid shape also increases.

# 6. Using value ranges with excludeRasterType
`excludeRasterType` can exclude raster cells based on their values. For categorical rasters such as CORINE Land Cover (CLC), multiple classes can be selected in a single value expression.

Instead of listing every value separately, you can also use compact range syntax.

Here, three groups of CLC classes are selected:

- `[1-2]` selects all classes from 1 to 2
- `12` selects class 12
- `[21-23]` selects all classes from 21 to 23

The comma `,` combines multiple selections. Square brackets `[ ]` include the endpoints of a range, while parentheses `( )` exclude them. For example, `[21-23)` includes 21 and 22, but not 23. And vice vers `(21-23]` selects 22 and 23, but not 21.

In [ ]:
ec = gl.ExclusionCalculator(region_path, srs=3035, pixelRes=100)

ec.excludeRasterType(
    source=clc_path,
    value="[1-2],12,[21-23]",
)

ec.draw()

# 7. Alternative exclusion approaches (include, invert, initialValue)

By default, a new `ExclusionCalculator` starts with all cells inside the region marked as eligible. With `initialValue=False`, this can be reversed so that all cells start as unavailable.

This allows eligibility to be approached from two directions. For example, suppose we want only the CLC classes `[1-2]`, `12`, and `[21-23]` to remain eligible.

The first approach starts with nothing available and explicitly includes these classes:

In [ ]:
# Start with everything unavailable.
ec = gl.ExclusionCalculator(
    region=region_path,
    srs=3035,
    pixelRes=100,
    initialValue=False,
)

# Include only the selected land cover classes.
ec.excludeRasterType(
    source=clc_path,
    value="[1-2],12,[21-23]",
    mode="include",
)

print("Percent available with include mode:", ec.percentAvailable)
ec.draw()

Alternatively, we can start with everything available and use `invert=True` to exclude everything except the selected classes:

In [ ]:
# Start with everything available.
ec = gl.ExclusionCalculator(
    region=region_path,
    srs=3035,
    pixelRes=100,
)

# Invert the selection, then exclude it.
# This excludes all cells NOT in the selected classes,
# so only the same selected classes remain eligible.
ec.excludeRasterType(
    source=clc_path,
    value="[1-2],12,[21-23]",
    invert=True,
)

print("Percent available with invert=True:", ec.percentAvailable)
ec.draw()

**Why do these look similar?** 

Both approaches leave the same CLC classes eligible, but they get there differently:

- `initialValue=False` + `mode="include"`: start with nothing and add the selected classes.
- `invert=True`: start with everything and remove everything except the selected classes.

# 8. Working with Digital Elevation Model (DEM) mosaics for elevation and slope exclusions
Terrain can affect whether an area is suitable for a specific use. A **Digital Elevation Model (DEM)** provides the elevation of the terrain and can also be used to derive additional information, such as slope and orientation.

In this example, we use DEM data to create three terrain-based exclusion criteria:

1. First, multiple DEM tiles are combined into one elevation raster covering the full study area with `rasterMosaic()`.
2. The elevation raster is used directly to exclude areas above 500 m.
3. The DEM mosaic is used to calculate the slope, and areas steeper than 15° are excluded.
4. Finally, the north–south gradient is calculated to identify and exclude north-facing slopes.

This demonstrates how a single DEM can be used both directly as an exclusion raster and as a basis for deriving additional raster layers for an eligibility analysis.


DEM data for other regions can be downloaded from:

- `https://dwtkns.com/srtm30m`  








#### 8.1 Creating the DEM mosaic

In [ ]:
import os

dem_dir = os.path.dirname(gl._test_data_["aachenShapefile.shp"])
dem_raster = ec.region.extent.rasterMosaic(os.path.join(dem_dir, "*.hgt")) # .hgt files are used as DEM inputs
print(type(dem_raster)) # multiple DEM files merged into a single raster mosaic
gk.drawRaster(dem_raster, cbarTitle="Elevation [m]") # the mosaic is displayed with `gk.drawRaster()` as an elevation heatmap

#### 8.2 Excluding areas above 500 m

In [ ]:
# exclude areas above 500m elevation. Note, that here the hilly Eiffel region is mainly exluded in the region's south.
ec = gl.ExclusionCalculator(region_path, srs=3035, pixelRes=100)
ec.excludeRasterType(dem_raster, value="(500-]", resampleAlg="mode")
ec.draw()

#### 8.3 Excluding slopes above 15°

In [ ]:
# Exclude Raster - DEM Gradient Mosaic. Exlude slopes above 15 degrees.

dem_gradient_raster = gk.raster.mutateRaster(
    source=gk.raster.gradient(  # computes the gradient (slope) of a raster
        source=dem_raster, factor="latlonToM"
    ),  # factor to convert lat/lon degrees to meters
    processor=lambda x: np.degrees(np.arctan(x)),  # convert gradient to degrees
)

ec = gl.ExclusionCalculator(region_path, srs=3035, pixelRes=100)
ec.excludeRasterType(dem_gradient_raster, value=(15, None), resampleAlg="mode")

ec.draw()

#### 8.4 Excluding north-facing slopes

In [ ]:
# Exlude slopes with north orientation, as north facing slopes receive less radiation.

dem_ns_gradient_raster = gk.raster.mutateRaster(
    source=gk.raster.gradient(source=dem_raster, factor="latlonToM", mode="north-south"),
    processor=lambda matrix: np.degrees(np.arctan(matrix)),
)

ec = gl.ExclusionCalculator(region_path, srs=3035, pixelRes=100)

# exclude all north facing slopes (values > 0 degrees)
ec.excludeRasterType(dem_ns_gradient_raster, value=(0, None), resampleAlg="mode")

ec.draw()

# 9. ExcludeVectorType
This section shows how `excludeVectorType()` can be used to exclude areas from vector datasets.

Use where clause to exclude IUCN_CAT='V' (IUCN protected area management category, category V: Protected Landscape):

In [ ]:
#  Exclude Vector - Simple source w/ where clause

ec = gl.ExclusionCalculator(region=gk._test_data_["aachenShapefile.shp"], srs="LAEA", pixelRes=50, padExtent=5000)

ec.excludeVectorType(
    source=r"data/WDPA_aachen_bbox.shp",
    where="IUCN_CAT='V'",
)

ec.draw()

The following example uses `resolutionDiv` to rasterize vector features at a finer internal resolution.

In [ ]:
# Exclude Vector - resolutionDiv
ec = gl.ExclusionCalculator(region=gk._test_data_["aachenShapefile.shp"], srs="LAEA", pixelRes=50, padExtent=5000)

ec.excludeVectorType(
    source=r"data/WDPA_aachen_bbox.shp",
    where="IUCN_CAT='V'",
    buffer=400,
    resolutionDiv=8,
    intermediate="tmp/wdpa_V.tif",  # stores the rasterized exclusion result for these exact arguments, so the same exclusion can be reused later instead of recalculated
)
ec.draw()

`ec.save()` exports the current availability result of the ExclusionCalculator as a GeoTIFF raster

In [ ]:
ec.save?

In [ ]:
ec.save("tmp/test_resolution_div_8.tif")

# 10. Visualization Methods

In [ ]:
# regular drawing with custom colors and legend position
ec.draw(
    goodColor="blue",
    excludedColor=(1.0, 0.2, 0.0, 0.6),
    legendargs={"loc": "upper right"},
)

Additional spatial layers can also be drawn on top of the exclusion result.
In the example below, major roads are added as a vector overlay.

In [ ]:
road_path = gl._test_data_["aachenRoads.shp"]

roads = gk.vector.extractFeatures(
    road_path,
    geom=ec.region.geometry,
)

Before drawing the roads, we inspect the available road categories in the type column. These values can then be used to build the where filter for selecting only major roads.

In [ ]:
roads.type.unique()

In [ ]:
ax = ec.draw()
major_roads = gk.vector.extractFeatures(
    road_path,
    where="type='motorway' OR type='primary' OR type='trunk'",
)
gk.drawGeoms(major_roads, ax=ax, color="k", linestyle="--", srs=ec.region.srs)
plt.show()

`drawWithSmopyBasemap()` (cf. section 3) can be used to visualize roads in its geographic context.

In [ ]:
ax = ec.drawWithSmopyBasemap(
    zoom=9,
    legendargs={"loc": "upper right"},
)
gk.drawGeoms(major_roads, ax=ax, color="k", linestyle="--", srs=gk.srs.EPSG3857)
plt.show()